# SpecRoute - Remote Target Model Server

Run this notebook in Google Colab or Kaggle to host the target model (e.g., Qwen2.5-7B-Instruct).
It exposes a verification API via ngrok that the local SpecRoute client connects to.

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate fastapi uvicorn pyngrok pydantic

In [ ]:
# Configuration - edit these values
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"    # Target model
TOKENIZER_NAME = "Qwen/Qwen2.5-7B-Instruct"  # Must match local client tokenizer
NGROK_AUTH_TOKEN = "2IYFAAulfcHwLVXcIlpFsarQphg_5Km1K5q3Q6RTM6CPg9geF"     # Get from https://dashboard.ngrok.com
PORT = 8001

In [ ]:
import logging
import random
from typing import Optional

import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from transformers.cache_utils import DynamicCache
except ImportError:
    DynamicCache = None

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("specroute.target")


# Protocol messages (inline to keep notebook self-contained)
class SessionInitRequest(BaseModel):
    session_id: str
    prompt_token_ids: list[int]
    tokenizer_name: str

class SessionInitResponse(BaseModel):
    session_id: str
    ready: bool
    cache_position: int

class VerifyRequest(BaseModel):
    session_id: str
    draft_token_ids: list[int]
    draft_probs: list[float]
    cache_position: int

class VerifyResponse(BaseModel):
    accepted_count: int
    corrective_token_id: int
    accepted_token_ids: list[int]
    target_probs: list[float]
    new_cache_position: int


def _truncate_kv(past_key_values, max_len):
    """Truncate a KV-cache (DynamicCache or tuple-of-tuples) to max_len."""
    if DynamicCache is not None and isinstance(past_key_values, DynamicCache):
        new_cache = DynamicCache()
        for layer_idx in range(len(past_key_values.key_cache)):
            new_cache.update(
                past_key_values.key_cache[layer_idx][:, :, :max_len, :],
                past_key_values.value_cache[layer_idx][:, :, :max_len, :],
                layer_idx,
            )
        return new_cache
    return tuple(
        (k[:, :, :max_len, :], v[:, :, :max_len, :])
        for k, v in past_key_values
    )


class TargetModelServer:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.sessions = {}

    def load_model(self, model_name, tokenizer_name, device="cuda"):
        log.info("Loading target model: %s", model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )
        self.model.eval()
        log.info("Target model loaded on %s", device)

    def init_session(self, req):
        input_ids = torch.tensor([req.prompt_token_ids], device=self.model.device)
        with torch.no_grad():
            out = self.model(input_ids=input_ids, use_cache=True)
        self.sessions[req.session_id] = {
            "past_key_values": out.past_key_values,
            "cache_len": len(req.prompt_token_ids),
            "last_logits": out.logits[0, -1, :],
        }
        return SessionInitResponse(
            session_id=req.session_id, ready=True,
            cache_position=len(req.prompt_token_ids),
        )

    def verify(self, req):
        session = self.sessions.get(req.session_id)
        if not session:
            raise ValueError(f"Unknown session: {req.session_id}")

        past_kv = session["past_key_values"]
        last_logits = session["last_logits"]
        draft_ids = req.draft_token_ids
        draft_probs = req.draft_probs
        n_draft = len(draft_ids)
        temperature = 0.7

        input_ids = torch.tensor([draft_ids], device=self.model.device)
        with torch.no_grad():
            out = self.model(input_ids=input_ids, past_key_values=past_kv, use_cache=True)

        all_logits = torch.stack(
            [last_logits] + [out.logits[0, i, :] for i in range(n_draft - 1)]
        )

        if temperature > 0:
            target_probs_full = torch.softmax(all_logits / temperature, dim=-1)
        else:
            target_probs_full = torch.softmax(all_logits, dim=-1)

        target_probs_for_draft = [target_probs_full[i, tid].item() for i, tid in enumerate(draft_ids)]

        accepted_count = 0
        for i in range(n_draft):
            p, q = target_probs_for_draft[i], draft_probs[i]
            if q <= 0:
                break
            if p >= q or random.random() < (p / q):
                accepted_count += 1
            else:
                break

        if accepted_count == n_draft:
            next_logits = out.logits[0, -1, :]
        else:
            next_logits = all_logits[accepted_count]

        if temperature <= 0:
            corrective_id = next_logits.argmax().item()
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            corrective_id = torch.multinomial(probs, num_samples=1).item()

        trunc_len = session["cache_len"] + accepted_count
        trunc_kv = _truncate_kv(out.past_key_values, trunc_len)

        corr_input = torch.tensor([[corrective_id]], device=self.model.device)
        with torch.no_grad():
            corr_out = self.model(input_ids=corr_input, past_key_values=trunc_kv, use_cache=True)

        session["past_key_values"] = corr_out.past_key_values
        session["cache_len"] = trunc_len + 1
        session["last_logits"] = corr_out.logits[0, -1, :]

        log.info("[Speculation] %d/%d accepted | corrective=%d (%s)",
                 accepted_count, n_draft, corrective_id,
                 self.tokenizer.decode([corrective_id]))

        return VerifyResponse(
            accepted_count=accepted_count,
            corrective_token_id=corrective_id,
            accepted_token_ids=draft_ids[:accepted_count],
            target_probs=target_probs_for_draft,
            new_cache_position=session["cache_len"],
        )


target_server = TargetModelServer()
app = FastAPI(title="SpecRoute Target Server")

@app.post("/api/session/init")
async def session_init(req: SessionInitRequest):
    try:
        return target_server.init_session(req).model_dump()
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/verify")
async def verify(req: VerifyRequest):
    try:
        return target_server.verify(req).model_dump()
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    return {"status": "ok"}

print("Server code defined.")

Server code defined.


In [ ]:
# Load the target model
target_server.load_model(MODEL_NAME, TOKENIZER_NAME, device="cuda")
print(f"Model {MODEL_NAME} loaded successfully.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model Qwen/Qwen2.5-7B-Instruct loaded successfully.


In [ ]:
# Start ngrok tunnel and run the server
import asyncio
import nest_asyncio
from pyngrok import ngrok, exception as pyngrok_exception
import uvicorn

nest_asyncio.apply()

try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
except pyngrok_exception.PyngrokNgrokError:
    # Already set in the configuration file
    pass

tunnel = ngrok.connect(PORT)
print(f"\n{'='*60}")
print(f"SpecRoute Target Server is live!")
print(f"Public URL: {tunnel.public_url}")
print(f"\nSet this in your local config.yaml:")
print(f"  remote:")
print(f"    url: \"{tunnel.public_url}\"")
print(f"{'='*60}\n")

# Run uvicorn on the running loop
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, loop="asyncio")
server = uvicorn.Server(config)

# Run the server inside the existing event loop
asyncio.get_event_loop().run_until_complete(server.serve())

INFO:     Started server process [568]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)



SpecRoute Target Server is live!
Public URL: https://1fb3-34-143-181-142.ngrok-free.app

Set this in your local config.yaml:
  remote:
    url: "https://1fb3-34-143-181-142.ngrok-free.app"

